# 01 — Unique events: Variable 1nt vs Strict

This notebook identifies which splicing events are detected by SUPPA
in variable 1nt mode but NOT in strict mode, at the generateEvents level.

This comparison is purely structural — it does not involve expression
or statistical testing. We are comparing the event catalogues produced
by generateEvents with different boundary parameters.

**Events analysed:** A3, A5, RI  
**Comparison:** Strict IOE vs Variable 1nt IOE

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# IOE file paths
IOE_STRICT = "/Users/gricey/Desktop/Internship/data/output_strict/events"
IOE_VAR1   = "/Users/gricey/Desktop/Internship/data/output_variable_1nt/events"

# Events of interest
EVENTS = ["A3", "A5", "RI"]

# IOE filename suffixes
SUFFIX_STRICT = "strict"
SUFFIX_VAR1   = "variable_1"

# Output directory
OUTPUT_DIR = "/Users/gricey/Desktop/Internship/data/ioe_diff"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Helper functions ---

def style_table(df, caption=""):
    """Apply consistent blue-header styling to a DataFrame for display."""
    styled = df.style\
        .set_properties(**{
            "font-size": "12px",
            "font-weight": "bold",
            "border": "1px solid #ddd",
            "padding": "6px 12px",
            "text-align": "center"
        })\
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#2196F3"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("padding", "6px 12px"),
                ("text-align", "center")
            ]},
            {"selector": "tr:nth-child(even)", "props": [
                ("background-color", "#f2f2f2")
            ]},
        ])\
        .hide(axis="index")
    if caption:
        styled = styled.set_caption(caption)
    return styled

def export_table_png(df, filepath, title=""):
    """Export a DataFrame as a styled PNG image."""
    fig, ax = plt.subplots(figsize=(len(df.columns) * 1.8, len(df) * 0.6 + 0.8))
    ax.axis("off")
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.6)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2196F3")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f2f2f2")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#dddddd")
    if title:
        ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {filepath}")

print("Setup done!")

## Step 1 — Load IOE files

We load the IOE files produced by generateEvents for both strict and
variable 1nt modes. Each file contains one row per detected splicing event.

In [ ]:
ioe_data = {}

for event in EVENTS:
    strict_path = f"{IOE_STRICT}/events_{event}_{SUFFIX_STRICT}.ioe"
    var1_path   = f"{IOE_VAR1}/events_{event}_{SUFFIX_VAR1}.ioe"

    ioe_data[event] = {
        "strict": pd.read_csv(strict_path, sep="\t"),
        "var1":   pd.read_csv(var1_path,   sep="\t"),
    }

    print(f"{event} — strict: {len(ioe_data[event]['strict'])} events, "
          f"variable 1nt: {len(ioe_data[event]['var1'])} events")

## Step 2 — Count summary (Table 1)

Simple count of how many events each mode detects per event type,
and the raw difference V1 - S.

In [ ]:
rows = []
for event in EVENTS:
    n_strict = len(ioe_data[event]["strict"])
    n_var1   = len(ioe_data[event]["var1"])
    rows.append({
        "Event":          event,
        "Strict":         n_strict,
        "Variable 1nt":   n_var1,
        "V1 - S":         n_var1 - n_strict,
    })

df_counts = pd.DataFrame(rows)

display(style_table(df_counts, "Table 1 — IOE event counts: Strict vs Variable 1nt"))

export_table_png(
    df_counts,
    "../../figures/plots/table1_ioe_counts.png",
    title="Table 1 — IOE event counts: Strict vs Variable 1nt"
)

## Step 3 — Find unique events in Variable 1nt (Table 2)

We extract the gene IDs from each IOE file and find which genes
are present in variable 1nt but completely absent in strict.

Note: we compare at the gene level (ENSMUSG ID) because strict and
variable mode use different coordinate systems for the event boundaries.

In [ ]:
def get_gene_ids(ioe_df):
    return set(ioe_df["gene_id"].unique())

unique_results = {}
rows_unique = []

for event in EVENTS:
    strict_genes = get_gene_ids(ioe_data[event]["strict"])
    var1_genes   = get_gene_ids(ioe_data[event]["var1"])

    unique_genes = var1_genes - strict_genes

    unique_results[event] = {
        "unique_genes": unique_genes,
        "ioe_unique":   ioe_data[event]["var1"][
            ioe_data[event]["var1"]["gene_id"].isin(unique_genes)
        ]
    }

    rows_unique.append({
        "Event":              event,
        "Strict genes":       len(strict_genes),
        "Variable 1nt genes": len(var1_genes),
        "Unique to Var1nt":   len(unique_genes),
    })

df_unique = pd.DataFrame(rows_unique)

display(style_table(df_unique, "Table 2 — Genes unique to Variable 1nt vs Strict"))

export_table_png(
    df_unique,
    "../../figures/plots/table2_unique_genes.png",
    title="Table 2 — Genes unique to Variable 1nt vs Strict"
)

## Step 4 — Export unique events to IOE files

We save the IOE entries for the unique genes to new files.
These are the files to load into IGV for visualisation.

In [ ]:
for event in EVENTS:
    ioe_out  = f"{OUTPUT_DIR}/{event}_var1_unique.ioe"
    gene_out = f"{OUTPUT_DIR}/{event}_var1_unique_genes.txt"

    # Save IOE file
    unique_results[event]["ioe_unique"].to_csv(ioe_out, sep="\t", index=False)

    # Save gene list
    with open(gene_out, "w") as f:
        for gene in sorted(unique_results[event]["unique_genes"]):
            f.write(gene + "\n")

    print(f"{event} → {len(unique_results[event]['ioe_unique'])} IOE entries, "
          f"{len(unique_results[event]['unique_genes'])} unique genes")